# dYdX collector catalog -> pandas

Reads the collector's `ParquetDataCatalog` output directly through Nautilus's own catalog API, which already decodes the fixed-point `Price`/`Quantity` blobs and derives `instrument_id` -- no manual byte-decoding needed (unlike querying the raw Parquet files with a generic tool). `to_dict()` is a staticmethod on each `Data`/`Instrument` class, so it's called as `TradeTick.to_dict(t)`, not `t.to_dict()`.

Catalog reads below are always time-bounded (`start=`/`end=` in nanoseconds) rather than pulling a full unbounded history into memory -- required by NFR3/AD-6, since this catalog grows continuously as the collector keeps running.

In [ ]:
import pandas as pd

from nautilus_trader.model.data import TradeTick
from nautilus_trader.model.instruments import CryptoPerpetual
from nautilus_trader.persistence.catalog import ParquetDataCatalog


catalog = ParquetDataCatalog("../catalog")  # relative to this notebook's directory
instrument_ids = [i.id.value for i in catalog.instruments()]
instrument_ids

In [ ]:
instruments_df = pd.DataFrame([CryptoPerpetual.to_dict(i) for i in catalog.instruments()])
instruments_df

In [ ]:
import time

# Bound the query -- an unbounded catalog.trade_ticks() call materializes every
# trade tick ever collected for this instrument into memory (NFR3/MEM-01).
# Note: per-instrument retain_hours pruning (dydx_collector/collector.py) may make the
# actual coverage shorter than LOOKBACK_HOURS if this instrument has a tighter retention.
LOOKBACK_HOURS = 24
start_ns = time.time_ns() - LOOKBACK_HOURS * 3_600 * 1_000_000_000

instrument_id = instrument_ids[0]
trades = catalog.trade_ticks(instrument_ids=[instrument_id], start=start_ns)

if not trades:
    print(f"No trades for {instrument_id} in the last {LOOKBACK_HOURS}h.")
trades_df = pd.DataFrame([TradeTick.to_dict(t) for t in trades])
if not trades_df.empty:
    # price/size come back as exact strings (avoids float precision loss in storage);
    # cast to float here for plotting/arithmetic.
    trades_df["price"] = trades_df["price"].astype(float)
    trades_df["size"] = trades_df["size"].astype(float)
    trades_df["ts_event"] = pd.to_datetime(trades_df["ts_event"], unit="ns")
    trades_df = trades_df.set_index("ts_event").sort_index()
trades_df

In [ ]:
if not trades_df.empty:
    trades_df["price"].plot(title=instrument_id)

## Reusing an existing indicator (never redefine inline)

Indicators are implemented once in `ml_signals.indicators` and imported here unmodified -- the same classes `metrics_computer.py`/`chart_data.py` already use for the live dashboard, and later for backtest/live contexts (FR-10). A new indicator always gets added to that module, never redefined inline in a notebook cell.

`Microprice` is fed from top-of-book state derived by replaying `OrderBookDelta` events -- this catalog does not store `QuoteTick` data (see `troll/CLAUDE.md`'s 1s-snapshot signal architecture), so `book_features.top_of_book_series` (a pure, side-effect-free utility per AD-4, already used by `metrics_computer.py`) is reused rather than reinvented here.

**Raw order-book-delta capture is opt-in per instrument** (`dydx_collector/config.py`'s `store_order_book_deltas`, default `False`) -- most instruments have none stored at all. The cell below tries each instrument until it finds one with deltas in the window, rather than assuming the first instrument has them.

In [ ]:
import sys

if "../.." not in sys.path:
    sys.path.insert(0, "../..")  # troll/ -- makes `ml_signals` importable (same relative-path convention as the catalog path above)

from nautilus_trader.model.identifiers import InstrumentId

from ml_signals.book_features import top_of_book_series
from ml_signals.indicators import Microprice

# Raw deltas are far higher-volume than trade ticks -- reusing the 24h trade-tick window
# here would be slow/heavy (metrics_computer.py uses 60s for this same replay;
# chart_data.py's dashboard chart defaults to 4h). Use a tighter, purpose-sized window.
DELTA_LOOKBACK_HOURS = 4
delta_start_ns = time.time_ns() - DELTA_LOOKBACK_HOURS * 3_600 * 1_000_000_000

# Try each instrument until one actually has deltas in the window -- instrument_ids[0] is
# not necessarily among the opt-in delta-capture set (see markdown above).
deltas = []
delta_instrument_id = None
for iid in instrument_ids:
    deltas = catalog.order_book_deltas(instrument_ids=[iid], start=delta_start_ns)
    if deltas:
        delta_instrument_id = iid
        break

if not deltas:
    print(f"No order-book deltas for any instrument in the last {DELTA_LOOKBACK_HOURS}h -- "
          "raw delta capture is opt-in per instrument (see dydx_collector/config.py).")
else:
    microprice = Microprice()
    micro_series = []
    prev_ts = None
    # Matches metrics_computer.py's _BOOK_GAP_NS -- a gap this large means the book was
    # desynced/reconnecting (see collector.py's resync watchdog); treat the next update as a
    # fresh start instead of splicing a discontinuity into one continuous series.
    _GAP_NS = 3_000_000_000

    for ts, bid_p, bid_s, ask_p, ask_s in top_of_book_series(deltas, InstrumentId.from_str(delta_instrument_id)):
        if prev_ts is not None and ts - prev_ts > _GAP_NS:
            microprice.reset()
        microprice.update_raw(bid_p, bid_s, ask_p, ask_s)
        if microprice.initialized:  # update_raw is a no-op on zero total size -- skip the stale/uninitialized value
            micro_series.append((ts, microprice.value))
        prev_ts = ts

    if not micro_series:
        print(f"{delta_instrument_id} had deltas but no valid top-of-book state in window.")
    else:
        micro_df = pd.DataFrame(micro_series, columns=["ts_event", "microprice"])
        micro_df["ts_event"] = pd.to_datetime(micro_df["ts_event"], unit="ns")
        micro_df.set_index("ts_event").sort_index()["microprice"].plot(title=f"{delta_instrument_id} microprice")